# Stop, Stream & NodeContext

Three new capabilities added to the hypergraph framework:

| Feature | What it does |
|---|---|
| **NodeContext** | Injected via type-hint detection (FastAPI pattern). Gives nodes access to runtime context. |
| **Cooperative Stop** | `runner.stop(workflow_id)` sets a signal that nodes can check via `ctx.stop_requested`. |
| **Streaming** | `ctx.stream(chunk)` emits `StreamingChunkEvent` for live UI preview. |

This notebook demonstrates every feature with runnable examples.

In [ ]:
import asyncio
import threading
from unittest.mock import MagicMock

from hypergraph import AsyncRunner, Graph, SyncRunner, node
from hypergraph.runners._shared.node_context import NodeContext

## 1. NodeContext Injection

Add `ctx: NodeContext` to any node function. The framework:
- **Excludes** it from `node.inputs` (it's not a data dependency)
- **Injects** it automatically at execution time
- Works with any parameter name, as long as the type hint is `NodeContext`

In [ ]:
@node(output_name="result")
def my_node(x: int, ctx: NodeContext) -> int:
    print(f"  stop_requested = {ctx.stop_requested}")
    print(f"  stream callable = {callable(ctx.stream)}")
    return x * 2


# NodeContext is NOT a graph input
print(f"node.inputs = {my_node.inputs}")  # ('x',) -- no 'ctx'
print(f"graph inputs = {Graph([my_node]).inputs.all}")

In [ ]:
# Run it -- context is injected automatically
runner = SyncRunner()
result = runner.run(Graph([my_node]), x=5)
print(f"result = {result['result']}")

### Backward compatible

Nodes without `NodeContext` work exactly as before. No changes needed.

In [ ]:
@node(output_name="doubled")
def plain_node(x: int) -> int:
    return x * 2


result = SyncRunner().run(Graph([plain_node]), x=7)
print(f"plain_node result = {result['doubled']}")

## 2. Streaming: `ctx.stream(chunk)`

Call `ctx.stream(chunk)` to emit `StreamingChunkEvent`s in real time.
This is a side-channel -- it does NOT affect the node's return value.

In [ ]:
from hypergraph import StreamingChunkEvent


class ChunkCollector:
    """Event processor that captures streaming chunks."""

    def __init__(self):
        self.chunks = []

    def on_event(self, event):
        if isinstance(event, StreamingChunkEvent):
            self.chunks.append(event.chunk)

    def shutdown(self):
        pass

In [ ]:
@node(output_name="response")
def stream_demo(ctx: NodeContext) -> str:
    words = ["Hello", " ", "from", " ", "hypergraph", "!"]
    response = ""
    for word in words:
        ctx.stream(word)  # emit chunk for live preview
        response += word
    return response


collector = ChunkCollector()
result = SyncRunner().run(Graph([stream_demo]), event_processors=[collector])

print(f"Final return value: {result['response']!r}")
print(f"Streamed chunks:   {collector.chunks}")

### Async streaming works identically

In [ ]:
@node(output_name="response")
async def async_stream(ctx: NodeContext) -> str:
    tokens = ["async", "-", "streaming", "-", "works"]
    response = ""
    for token in tokens:
        ctx.stream(token)
        response += token
        await asyncio.sleep(0.01)
    return response


collector = ChunkCollector()
result = await AsyncRunner().run(Graph([async_stream]), event_processors=[collector])

print(f"Final: {result['response']!r}")
print(f"Chunks: {collector.chunks}")

## 3. Cooperative Stop

The stop mechanism is **cooperative**: `runner.stop(workflow_id)` sets a signal,
and nodes check `ctx.stop_requested` to decide when to break.

This preserves partial results (no `Task.cancel()`, no lost work).

### 3a. Basic stop mid-stream

In [ ]:
@node(output_name="response")
async def llm_generate(ctx: NodeContext) -> str:
    """Simulates an LLM generating 100 tokens. Checks stop per token."""
    response = ""
    for i in range(100):
        if ctx.stop_requested:
            break
        response += f"token-{i} "
        await asyncio.sleep(0.001)
    return response


runner = AsyncRunner()


# Stop after 30ms
async def stop_soon():
    await asyncio.sleep(0.03)
    runner.stop("demo", info={"reason": "user clicked stop"})


task = asyncio.create_task(stop_soon())
result = await runner.run(Graph([llm_generate]), workflow_id="demo")
await task

tokens = result["response"].strip().split()
print(f"Stopped:  {result.stopped}")
print(f"Status:   {result.status}")
print(f"Tokens:   {len(tokens)} / 100 (partial)")
print(f"Preview:  {result['response'][:60]}...")

### 3b. Stop from a thread (SyncRunner)

In [ ]:
import time


@node(output_name="result")
def slow_compute(ctx: NodeContext) -> str:
    result = ""
    for i in range(1000):
        if ctx.stop_requested:
            break
        result += f"step-{i} "
        time.sleep(0.0001)
    return result


sync_runner = SyncRunner()


def stop_from_thread():
    time.sleep(0.02)
    sync_runner.stop("sync-demo")


t = threading.Thread(target=stop_from_thread)
t.start()
result = sync_runner.run(Graph([slow_compute]), workflow_id="sync-demo")
t.join()

steps = result["result"].strip().split()
print(f"Stopped: {result.stopped}")
print(f"Status:  {result.status}")
print(f"Steps:   {len(steps)} / 1000")

### 3c. Stop without checkpointer

Stop works purely in-memory. No checkpointer needed for basic use.

In [ ]:
@node(output_name="result")
async def counting_node(ctx: NodeContext) -> str:
    for i in range(100):
        if ctx.stop_requested:
            return f"stopped at {i}"
        await asyncio.sleep(0.001)
    return "completed all 100"


runner = AsyncRunner()  # no checkpointer


async def stop_it():
    await asyncio.sleep(0.02)
    runner.stop("mem")


task = asyncio.create_task(stop_it())
result = await runner.run(Graph([counting_node]), workflow_id="mem")
await task

print(f"result:  {result['result']}")
print(f"stopped: {result.stopped}")
print(f"status:  {result.status}")

## 4. `WorkflowAlreadyRunningError`

Each `workflow_id` allows only one concurrent run. A second `run()` with the same ID raises.

In [ ]:
from hypergraph import WorkflowAlreadyRunningError


@node(output_name="x")
async def hang(ctx: NodeContext) -> int:
    await asyncio.sleep(10)
    return 1


runner = AsyncRunner()


async def try_second():
    await asyncio.sleep(0.02)
    try:
        await runner.run(Graph([hang]), workflow_id="unique-1")
    except WorkflowAlreadyRunningError as e:
        print(f"Caught: {e}")


async def cleanup():
    await asyncio.sleep(0.05)
    runner.stop("unique-1")


t1 = asyncio.create_task(try_second())
t2 = asyncio.create_task(cleanup())
await runner.run(Graph([hang]), workflow_id="unique-1")
await t1
await t2

## 5. Streaming + Stop Combined

Combine `ctx.stream()` and `ctx.stop_requested` for the canonical
LLM streaming pattern: stream chunks to the UI, stop when the user clicks stop.

In [ ]:
from hypergraph import StopRequestedEvent


class FullCollector:
    """Collects both streaming chunks and stop events."""

    def __init__(self):
        self.chunks = []
        self.stop_events = []

    def on_event(self, event):
        if isinstance(event, StreamingChunkEvent):
            self.chunks.append(event.chunk)
        elif isinstance(event, StopRequestedEvent):
            self.stop_events.append(event)

    def shutdown(self):
        pass


@node(output_name="response")
async def stream_with_stop(ctx: NodeContext) -> str:
    response = ""
    for i in range(100):
        if ctx.stop_requested:
            break
        chunk = f"word-{i} "
        ctx.stream(chunk)
        response += chunk
        await asyncio.sleep(0.001)
    return response


collector = FullCollector()
runner = AsyncRunner()


async def stop_mid():
    await asyncio.sleep(0.03)
    runner.stop("stream-stop", info={"kind": "user_stop"})


task = asyncio.create_task(stop_mid())
result = await runner.run(
    Graph([stream_with_stop]),
    workflow_id="stream-stop",
    event_processors=[collector],
)
await task

print(f"Chunks streamed: {len(collector.chunks)}")
print(f"Stop events:     {len(collector.stop_events)}")
if collector.stop_events:
    print(f"Stop info:       {collector.stop_events[0].info}")
print(f"Final result:    {result['response'][:50]}...")

## 6. `complete_on_stop`: Inner Graphs Finish After Stop

When a `GraphNode` has `complete_on_stop=True`, its inner graph runs
all remaining supersteps after the stop signal fires.

Use case: LLM generates partial output, then a postprocessing node
saves the partial result before the workflow stops.

In [ ]:
execution_order = []


@node(output_name="partial")
async def generate(ctx: NodeContext) -> str:
    execution_order.append("generate")
    result = ""
    for i in range(100):
        if ctx.stop_requested:
            return f"partial-{i}-tokens"
        result += f"t{i} "
        await asyncio.sleep(0.001)
    return result


@node(output_name="final")
def postprocess(partial: str) -> str:
    execution_order.append("postprocess")
    return f"saved:{partial}"


# Inner graph with complete_on_stop=True: postprocess WILL run after stop
inner = Graph([generate, postprocess], name="pipeline").as_node(complete_on_stop=True)
outer = Graph([inner])

runner = AsyncRunner()


async def stop_gen():
    await asyncio.sleep(0.02)
    runner.stop("cos")


task = asyncio.create_task(stop_gen())
result = await runner.run(outer, workflow_id="cos")
await task

print(f"Execution order: {execution_order}")
print(f"Final value:     {result['final']}")
print(f"Stopped:         {result.stopped}")

### Without `complete_on_stop`: postprocess is skipped

In [ ]:
execution_order_no_cos = []


@node(output_name="partial")
async def generate2(ctx: NodeContext) -> str:
    execution_order_no_cos.append("generate")
    for _i in range(100):
        if ctx.stop_requested:
            return "partial"
        await asyncio.sleep(0.001)
    return "full"


@node(output_name="final")
def postprocess2(partial: str) -> str:
    execution_order_no_cos.append("postprocess")
    return f"saved:{partial}"


# Default: complete_on_stop=False
inner_no_cos = Graph([generate2, postprocess2], name="pipeline").as_node()

runner = AsyncRunner()


async def stop_gen2():
    await asyncio.sleep(0.02)
    runner.stop("no-cos")


task = asyncio.create_task(stop_gen2())
await runner.run(Graph([inner_no_cos]), workflow_id="no-cos")
await task

print(f"Execution order: {execution_order_no_cos}")
print("postprocess was SKIPPED because the inner graph broke at the superstep boundary")

## 7. Nested Stop Propagation

Stop signals propagate to nested graphs automatically via a parent chain.
Calling `runner.stop()` on the outer workflow stops inner graphs too.

In [ ]:
nested_saw_stop = {}


@node(output_name="inner_result")
async def inner_node(ctx: NodeContext) -> str:
    for _ in range(100):
        if ctx.stop_requested:
            nested_saw_stop["inner"] = True
            return "inner stopped"
        await asyncio.sleep(0.001)
    return "inner completed"


inner_graph = Graph([inner_node], name="inner").as_node()
outer_graph = Graph([inner_graph])

runner = AsyncRunner()


async def stop_outer():
    await asyncio.sleep(0.02)
    runner.stop("nested")


task = asyncio.create_task(stop_outer())
result = await runner.run(outer_graph, workflow_id="nested")
await task

print(f"Inner saw stop: {nested_saw_stop.get('inner', False)}")
print(f"Outer stopped:  {result.stopped}")
print(f"Inner result:   {result['inner_result']}")

## 8. Steering: Stop + Redirect

Stop one generation, start a fresh one with different input.
Common pattern: user changes their mind mid-generation.

In [ ]:
@node(output_name="response")
async def gen_topic(prompt: str, ctx: NodeContext) -> str:
    result = ""
    for i in range(100):
        if ctx.stop_requested:
            break
        result += f"{prompt}-{i} "
        await asyncio.sleep(0.001)
    return result


graph = Graph([gen_topic], name="gen")
runner = AsyncRunner()


# Turn 1: start generating about cats, then stop
async def stop_turn1():
    await asyncio.sleep(0.02)
    runner.stop("steer-1")


task = asyncio.create_task(stop_turn1())
r1 = await runner.run(graph, workflow_id="steer-1", prompt="cats")
await task

print(f"Turn 1 (stopped): {r1['response'][:40]}...")
print(f"Turn 1 stopped:   {r1.stopped}")

# Turn 2: redirect to dogs (fresh run, no stop)
r2 = await runner.run(graph, prompt="dogs")

print(f"\nTurn 2 (full):    {r2['response'][:40]}...")
print(f"Turn 2 stopped:   {r2.stopped}")
print(f"Turn 2 tokens:    {len(r2['response'].strip().split())}")

## 9. Batch Processing with Per-Item Stop

Process a large batch, checking `stop_requested` per item.
Partial results are preserved.

In [ ]:
@node(output_name="results")
async def process_batch(items: list, ctx: NodeContext) -> list:
    results = []
    for item in items:
        if ctx.stop_requested:
            break
        results.append(item * 2)
        await asyncio.sleep(0.001)
    return results


runner = AsyncRunner()


async def stop_batch():
    await asyncio.sleep(0.02)
    runner.stop("batch")


task = asyncio.create_task(stop_batch())
result = await runner.run(
    Graph([process_batch]),
    workflow_id="batch",
    items=list(range(1000)),
)
await task

print(f"Processed: {len(result['results'])} / 1000 items")
print(f"Stopped:   {result.stopped}")
print(f"First 5:   {result['results'][:5]}")

## 10. `StepRecord.partial` with Checkpointer

When using a checkpointer, stopped nodes get `partial=True` on their `StepRecord`.
This lets you distinguish partial from complete results in the checkpoint history.

In [ ]:
import tempfile

from hypergraph.checkpointers import SqliteCheckpointer


@node(output_name="result")
async def long_node(ctx: NodeContext) -> str:
    for _ in range(1000):
        if ctx.stop_requested:
            return "partial-output"
        await asyncio.sleep(0.001)
    return "full-output"


db_path = tempfile.mktemp(suffix=".db")
cp = SqliteCheckpointer(db_path)
runner = AsyncRunner(checkpointer=cp)


async def stop_later():
    await asyncio.sleep(0.1)
    runner.stop("cp-demo")


task = asyncio.create_task(stop_later())
await runner.run(Graph([long_node]), workflow_id="cp-demo")
await task

# Inspect the checkpoint
checkpoint = await cp.get_checkpoint("cp-demo")
for step in checkpoint.steps:
    print(f"  node={step.node_name}, status={step.status}, partial={step.partial}")

await cp.close()

## 11. Testing with Mocks (Plain Python)

Nodes that use `NodeContext` are easy to test without the framework.
Just pass a `MagicMock(spec=NodeContext)`.

In [ ]:
@node(output_name="result")
def stoppable_compute(x: int, ctx: NodeContext) -> int:
    if ctx.stop_requested:
        return 0
    ctx.stream(x)
    return x * 2


# Test normal execution
ctx = MagicMock(spec=NodeContext)
ctx.stop_requested = False
assert stoppable_compute(5, ctx=ctx) == 10
ctx.stream.assert_called_once_with(5)
print("Normal path: 5 -> 10, stream called")

# Test stop path
ctx2 = MagicMock(spec=NodeContext)
ctx2.stop_requested = True
assert stoppable_compute(5, ctx=ctx2) == 0
print("Stop path:   5 -> 0, early exit")

print("\nNo framework needed for unit testing!")

## 12. Edge Cases

### `stop()` before `run()` is a no-op

In [ ]:
@node(output_name="r")
async def simple(x: int) -> int:
    return x * 2


runner = AsyncRunner()
runner.stop("wf")  # no active run -- silently ignored

result = await runner.run(Graph([simple]), workflow_id="wf", x=5)
print(f"result:  {result['r']}")
print(f"stopped: {result.stopped}  (stop before run has no effect)")

### `stop()` on nonexistent workflow is a no-op

In [ ]:
runner = AsyncRunner()
runner.stop("does-not-exist")  # no error
print("No error raised")

## Summary

| Feature | API | Notes |
|---|---|---|
| NodeContext injection | `def my_node(x: int, ctx: NodeContext)` | Type-hint detection; excluded from inputs |
| Stop signal | `runner.stop(workflow_id, info=...)` | Cooperative; partial results preserved |
| Check stop | `ctx.stop_requested` | Read-only bool property |
| Stream chunks | `ctx.stream(chunk)` | Emits `StreamingChunkEvent`; no-op after stop |
| Stop status | `result.stopped`, `result.status == RunStatus.STOPPED` | |
| Complete on stop | `graph.as_node(complete_on_stop=True)` | Inner graph finishes all supersteps |
| Partial flag | `step_record.partial` | Auto-inferred, persisted in SQLite |
| Nested propagation | Automatic via contextvar parent chain | |
| One run per ID | `WorkflowAlreadyRunningError` | |
| Testing | `MagicMock(spec=NodeContext)` | No framework needed |